### Customer Support AI Pipeline

In [11]:
# Run this cell once to install the Cohere SDK
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "cohere>=5.0.0", "-q"])
print("✅  Dependencies installed")

✅  Dependencies installed


In [12]:
from config import COHERE_API_KEY, COHERE_MODEL, CATEGORIES, SENTIMENTS

print(f"  API key loaded  (ends with: …{COHERE_API_KEY[-4:]})")
print(f"  Model          : {COHERE_MODEL}")
print(f"  Categories     : {', '.join(CATEGORIES)}")
print(f"  Sentiments     : {', '.join(SENTIMENTS)}")

  API key loaded  (ends with: …p2M8)
  Model          : command-a-03-2025
  Categories     : Complaint, Refund/Return, Sales Inquiry, Delivery Question, Account/Technical Issue, General Query, Spam
  Sentiments     : Positive, Neutral, Negative


In [13]:
from display_utils import render_banner
from processor import CustomerMessageProcessor

render_banner()
processor = CustomerMessageProcessor()
print("\n Processor ready — pipeline initialised!")

[Processor] Connected to Cohere — model: command-a-03-2025

 Processor ready — pipeline initialised!


In [14]:
import time
from demo_messages import DEMO_MESSAGES
from display_utils import render_batch, render_summary_table

print(f"🔄  Processing {len(DEMO_MESSAGES)} messages …\n")

messages = [item["message"] for item in DEMO_MESSAGES]
results  = []

for i, msg in enumerate(messages, 1):
    print(f"  [{i}/{len(messages)}] {DEMO_MESSAGES[i-1]['tag']} …", end=" ")
    result = processor.process(msg)
    results.append(result)
    status = "✅" if result["status"] == "success" else "❌ UNSUCCESSFUL "
    print(status)
    time.sleep(0.3)   # gentle on the free-tier rate limit

print(f"\n All {len(messages)} messages processed!\n")
render_summary_table(results)

🔄  Processing 7 messages …

  [1/7] Complaint … ✅
  [2/7] Refund/Return … ✅
  [3/7] Sales Inquiry … ✅
  [4/7] Delivery Question … ✅
  [5/7] Account/Technical Issue … ✅
  [6/7] General Query (Positive) … ✅
  [7/7] Spam … ✅

 All 7 messages processed!



#,Message,Category,Sentiment
#1,"""I ordered a laptop from your website two weeks ago and it st…""",Delivery Question,❌ Negative
#2,"""Hi, I'd like to return a jacket I purchased last week. Unfor…""",Refund/Return,➖ Neutral
#3,"""Hello! I'm interested in your premium annual subscription. D…""",Sales Inquiry,➖ Neutral
#4,"""My tracking number shows my order is still 'In Transit' afte…""",Delivery Question,➖ Neutral
#5,"""I'm completely locked out of my account. I tried resetting m…""",Account/Technical Issue,❌ Negative
#6,"""Just wanted to drop a note to say how impressed I am with yo…""",General Query,✅ Positive
#7,"""CONGRATULATIONS!!! You have WON a FREE iPhone 15!!! CLICK HE…""",Spam,➖ Neutral


In [15]:
# Full detailed cards for each message
render_batch(results)

In [16]:
from display_utils import render_result

#  Change this to any customer message you want to test
# ── Example 1 : Complaint
custom_message = "Your customer service is a complete joke! I've been waiting 3 weeks for a simple replacement and nobody has bothered to follow up. I want this escalated to a manager immediately!"

print("🔄  Processing …")
result = processor.process(custom_message)
render_result(result)

# ── Example 2 : Refund / Return
custom_message = "Hi, I received my order yesterday but the wrong item was delivered. I ordered a blue hoodie in size M but got a red one in XL. Please guide me on how to return it and get the correct item."

print("🔄  Processing …")
result = processor.process(custom_message)
render_result(result)

🔄  Processing …


🔄  Processing …


In [17]:
# ── Example 3 : Sales Inquiry
custom_message = "Hello! I'm considering upgrading to your business plan for my team of 12 people. Could you tell me what features are included and whether you offer a free trial before we commit?"

print("🔄  Processing …")
result = processor.process(custom_message)
render_result(result)

# ── Example 4 : Account / Technical Issue
custom_message = "I keep getting an error saying 'session expired' every time I try to checkout. I've cleared my cache and tried three different browsers but the problem persists. Please help!"

print("🔄  Processing …")
result = processor.process(custom_message)
render_result(result)

# ── Example 5 : Positive / General
custom_message = "Just wanted to say thank you! The support agent I spoke with yesterday was incredibly helpful and resolved my issue within minutes. Best customer experience I've had in a long time!"

print("🔄  Processing …")
result = processor.process(custom_message)
render_result(result)

# ── Example 6 : Spam
custom_message = "YOU HAVE BEEN SELECTED for a FREE $1000 gift card!!! Click NOW >>> www.freegift.xyz <<< Only 3 spots remaining!!! Don't miss out!!!"

print("🔄  Processing …")
result = processor.process(custom_message)
render_result(result)

# ── Example 7 : Delivery Question
custom_message = "My order was supposed to arrive between Monday and Wednesday but today is Friday and there's still no sign of it. The tracking just says 'Out for Delivery' since 6am. What's going on?"

print("🔄  Processing …")
result = processor.process(custom_message)
render_result(result)

🔄  Processing …


🔄  Processing …


🔄  Processing …


🔄  Processing …


🔄  Processing …


In [18]:
import json

export_data = [
    {
        "id":               i + 1,
        "original_message": r["original_message"],
        "category":         r["category"],
        "sentiment":        r["sentiment"],
        "auto_reply":       r["auto_reply"],
        "status":           r["status"],
    }
    for i, r in enumerate(results)
]

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print(f"✅  Saved {len(export_data)} results to results.json")

✅  Saved 7 results to results.json


In [19]:
# ── Save All Results to CSV
import csv

# Collect all results
all_results = results + [result]  # demo results + last custom message result

# If you ran multiple custom messages, add them manually like:
# all_results = results + [result1, result2, result3, result4, result5, result6, result7]

csv_filename = "customer_support_results.csv"

with open(csv_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "original_message", "category", "sentiment", "auto_reply", "status"])
    writer.writeheader()
    for i, r in enumerate(all_results, start=1):
        writer.writerow({
            "id":               i,
            "original_message": r["original_message"],
            "category":         r["category"],
            "sentiment":        r["sentiment"],
            "auto_reply":       r["auto_reply"],
            "status":           r["status"],
        })

print(f"✅  Saved {len(all_results)} results → {csv_filename}")

✅  Saved 8 results → customer_support_results.csv
